# BILL_AMT1 Recovery Before and After Imputation

This experiment tests how Self-Healing Machine Learning (SHML) handles simulated missing values in `BILL_AMT1`. It compares clean validation performance, fixed training-median imputation, and the recovery selected by SHML.

F1 is the primary metric. Accuracy, precision, recall, and ROC-AUC show how other aspects of model performance change. The test partition is reserved.


In [2]:
import sys
from pathlib import Path

import numpy as np

cwd = Path.cwd()
project_root = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "src" / "run_pipeline.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Start the notebook kernel inside the SHML repository.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.run_pipeline import inject_missingness, select_recovery

## Dataset and Experimental Setup

The experiment uses the UCI Default of Credit Card Clients dataset in `data/working/default of credit card clients.xls`. The 30,000 observations and 23 predictors are split into 21,000 training, 4,500 validation, and 4,500 reserved test observations.

Fit the scaler and logistic regression model on the training partition. Compute the control median from that same partition. For the single-seed run, use seed 42 to mask 675 of the 4,500 validation values in `BILL_AMT1` (15%).

SHML flags missingness above 5% and evaluates zero imputation first. The existing safety policy checks data integrity and permits up to 10% relative loss in F1 and recall compared with clean validation performance. If zero imputation fails, SHML evaluates median imputation and abstains if both candidates fail.


In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from src.run_pipeline import DATA_PATH, FEATURE
from src.evaluation import evaluate_model

# Load original dataset
df = pd.read_excel(DATA_PATH, header=1, engine="xlrd")

# Separate predictors and target
target = "default payment next month"

X = df.drop(columns=["ID", target])
y = df[target]

# Reproduce the original SHML data split
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

# Fit preprocessing ONLY on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

# Train baseline model
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)
model.fit(X_train_scaled, y_train)

# Calculate healthy validation performance
healthy_metrics = evaluate_model(
    X_batch=X_val,
    y_true=y_val,
    model=model,
    scaler=scaler
)

training_median = float(X_train[FEATURE].median())

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Untouched test rows:", len(X_test))
print("Training BILL_AMT1 median:", training_median)
print("Healthy F1:", round(healthy_metrics["f1"], 6))

Training rows: 21000
Validation rows: 4500
Untouched test rows: 4500
Training BILL_AMT1 median: 22225.5
Healthy F1: 0.356983


In [4]:

# Define the missingness level
missingness_rate = 0.15

# Inject missing values into validation data
X_corrupted = inject_missingness(
    X_batch=X_val,
    rate=missingness_rate
)

# Count missing values
missing_count = X_corrupted["BILL_AMT1"].isna().sum()

# Calculate actual missingness percentage
actual_missingness = (
    missing_count / len(X_corrupted)
) * 100

print("Feature:", FEATURE)
print("Validation observations:", len(X_corrupted))
print("Missing observations:", missing_count)
print("Actual missingness:", round(actual_missingness, 2), "%")

# Verify the original validation data is unchanged
assert X_val["BILL_AMT1"].isna().sum() == 0

# Verify the corrupted dataset has expected missingness
assert missing_count == 675

print("Missingness injection successful!")


Feature: BILL_AMT1
Validation observations: 4500
Missing observations: 675
Actual missingness: 15.0 %
Missingness injection successful!


## Single-Seed Recovery Results

At 15% missingness, SHML detected the failure and accepted zero imputation.

| Metric | Healthy | Median Control | SHML Recovery |
|---|---:|---:|---:|
| Accuracy | 0.808667 | 0.808000 | 0.808667 |
| Precision | 0.694767 | 0.683473 | 0.684066 |
| Recall | 0.240201 | 0.245226 | 0.250251 |
| F1 | 0.356983 | 0.360947 | 0.366446 |
| ROC-AUC | 0.713025 | 0.708958 | 0.708272 |

Zero imputation increased F1 by 0.005499 and recall by 0.005025 compared with fixed median imputation. Against the clean baseline, F1 and recall increased while precision and ROC-AUC declined. The comparison measures performance after imputation. The experiment did not score the raw batch with missing values.


In [5]:

# Create an independent copy of corrupted validation data
X_control = X_corrupted.copy(deep=True)

# Apply a fixed preprocessing strategy
# Median comes from training data, not validation data
X_control[FEATURE] = X_control[FEATURE].fillna(
    training_median
)

# Verify missing values were handled
assert X_control[FEATURE].isna().sum() == 0

# Verify original corrupted data remains unchanged
assert X_corrupted[FEATURE].isna().sum() == 675

print("Control strategy: Fixed median preprocessing")
print("Training median:", training_median)
print("Remaining missing values:", X_control[FEATURE].isna().sum())


Control strategy: Fixed median preprocessing
Training median: 22225.5
Remaining missing values: 0


In [6]:

# Evaluate the fixed-median control
control_metrics = evaluate_model(
    X_batch=X_control,
    y_true=y_val,
    model=model,
    scaler=scaler
)

print("Healthy F1:", round(healthy_metrics["f1"], 6))
print("Control F1:", round(control_metrics["f1"], 6))

f1_change = (
    control_metrics["f1"] - healthy_metrics["f1"]
)

print("Control vs healthy F1 change:", round(f1_change, 6))


Healthy F1: 0.356983
Control F1: 0.360947
Control vs healthy F1 change: 0.003964


In [7]:

from src.detection import detect_feature_missingness
from src.diagnosis import diagnose_failure

# Detect missingness
detection = detect_feature_missingness(
    X_batch=X_corrupted,
    feature=FEATURE,
    reference_missingness=0.0,
    threshold=0.05
)

# Diagnose failure
diagnosis = diagnose_failure(detection)

# Select and evaluate SHML recovery
chosen, preferred, fallback, status = select_recovery(
    X_corrupted=X_corrupted,
    y_val=y_val,
    diagnosis=diagnosis,
    training_median=training_median,
    healthy_metrics=healthy_metrics,
    model=model,
    scaler=scaler
)

assert chosen is not None, "No safe recovery was accepted"

shml_metrics = chosen["metrics"]

print("Detection:", detection["failure_detected"])
print("Recovery status:", status)
print("Selected strategy:", chosen["action"])

print("Healthy F1:", round(healthy_metrics["f1"], 6))
print("Median control F1:", round(control_metrics["f1"], 6))
print("SHML recovery F1:", round(shml_metrics["f1"], 6))

print(
    "SHML vs median control:",
    round(shml_metrics["f1"] - control_metrics["f1"], 6)
)


Detection: True
Recovery status: PREFERRED_ACCEPTED
Selected strategy: Zero imputation
Healthy F1: 0.356983
Median control F1: 0.360947
SHML recovery F1: 0.366446
SHML vs median control: 0.005499


In [8]:

import pandas as pd

metrics_to_compare = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc"
]

comparison = pd.DataFrame({
    "Metric": metrics_to_compare,
    "Healthy": [
        healthy_metrics[m] for m in metrics_to_compare
    ],
    "Median Control": [
        control_metrics[m] for m in metrics_to_compare
    ],
    "SHML Recovery": [
        shml_metrics[m] for m in metrics_to_compare
    ]
})

comparison["SHML vs Control"] = (
    comparison["SHML Recovery"]
    - comparison["Median Control"]
)

comparison["SHML vs Healthy"] = (
    comparison["SHML Recovery"]
    - comparison["Healthy"]
)

print(comparison.round(6).to_string(index=False))


   Metric  Healthy  Median Control  SHML Recovery  SHML vs Control  SHML vs Healthy
 accuracy 0.808667        0.808000       0.808667         0.000667         0.000000
precision 0.694767        0.683473       0.684066         0.000593        -0.010702
   recall 0.240201        0.245226       0.250251         0.005025         0.010050
       f1 0.356983        0.360947       0.366446         0.005499         0.009463
  roc_auc 0.713025        0.708958       0.708272        -0.000686        -0.004753


In [9]:

import numpy as np
import pandas as pd

from src.detection import detect_feature_missingness
from src.diagnosis import diagnose_failure

results = []
missingness_rate = 0.15

for seed in range(20):
    rng = np.random.default_rng(seed)

    # Generate a different missingness mask for each seed
    X_corrupted_seed = X_val.copy(deep=True)

    positions = rng.choice(
        len(X_corrupted_seed),
        size=int(len(X_corrupted_seed) * missingness_rate),
        replace=False
    )

    X_corrupted_seed.iloc[
        positions,
        X_corrupted_seed.columns.get_loc(FEATURE)
    ] = np.nan

    # Fixed median preprocessing control
    X_median = X_corrupted_seed.copy(deep=True)
    X_median[FEATURE] = X_median[FEATURE].fillna(
        training_median
    )

    median_metrics = evaluate_model(
        X_batch=X_median,
        y_true=y_val,
        model=model,
        scaler=scaler
    )

    # SHML detection and diagnosis
    detection = detect_feature_missingness(
        X_batch=X_corrupted_seed,
        feature=FEATURE,
        reference_missingness=0.0,
        threshold=0.05
    )

    diagnosis = diagnose_failure(detection)

    # SHML recovery-selection workflow
    chosen, preferred, fallback, status = select_recovery(
        X_corrupted=X_corrupted_seed,
        y_val=y_val,
        diagnosis=diagnosis,
        training_median=training_median,
        healthy_metrics=healthy_metrics,
        model=model,
        scaler=scaler
    )

    results.append({
        "seed": seed,
        "median_f1": median_metrics["f1"],
        "shml_f1": chosen["metrics"]["f1"] if chosen else np.nan,
        "shml_status": status,
        "selected_action": chosen["action"] if chosen else None
    })

results_df = pd.DataFrame(results)

results_df["f1_improvement"] = (
    results_df["shml_f1"] - results_df["median_f1"]
)

print(results_df.to_string(index=False))

print("\nMean median-control F1:",
      round(results_df["median_f1"].mean(), 6))

print("Mean SHML F1:",
      round(results_df["shml_f1"].mean(), 6))

print("Mean F1 difference:",
      round(results_df["f1_improvement"].mean(), 6))

print("Positive differences:",
      (results_df["f1_improvement"] > 0).sum(),
      "out of", len(results_df))

print("Abstentions:",
      (results_df["shml_status"] == "NO_SAFE_RECOVERY_ABSTAIN").sum())


 seed  median_f1  shml_f1        shml_status selected_action  f1_improvement
    0   0.370588 0.372807 PREFERRED_ACCEPTED Zero imputation        0.002219
    1   0.369276 0.373255 PREFERRED_ACCEPTED Zero imputation        0.003979
    2   0.362287 0.360947 PREFERRED_ACCEPTED Zero imputation       -0.001340
    3   0.362963 0.366986 PREFERRED_ACCEPTED Zero imputation        0.004023
    4   0.365782 0.370968 PREFERRED_ACCEPTED Zero imputation        0.005186
    5   0.362426 0.366838 PREFERRED_ACCEPTED Zero imputation        0.004412
    6   0.365926 0.369118 PREFERRED_ACCEPTED Zero imputation        0.003192
    7   0.363501 0.369004 PREFERRED_ACCEPTED Zero imputation        0.005502
    8   0.359050 0.364576 PREFERRED_ACCEPTED Zero imputation        0.005525
    9   0.365655 0.370588 PREFERRED_ACCEPTED Zero imputation        0.004933
   10   0.366593 0.372161 PREFERRED_ACCEPTED Zero imputation        0.005568
   11   0.365385 0.366176 PREFERRED_ACCEPTED Zero imputation        0.000792

In [10]:

from scipy import stats
import numpy as np

# Paired F1 differences across identical missingness masks
differences = results_df["f1_improvement"].dropna()

n = len(differences)
mean_diff = differences.mean()
std_diff = differences.std(ddof=1)
standard_error = std_diff / np.sqrt(n)

# 95% confidence interval using Student's t distribution
ci_low, ci_high = stats.t.interval(
    confidence=0.95,
    df=n - 1,
    loc=mean_diff,
    scale=standard_error
)

print("Number of seeds:", n)
print("Mean F1 improvement:", round(mean_diff, 6))
print("Standard deviation:", round(std_diff, 6))
print("95% CI:", (round(ci_low, 6), round(ci_high, 6)))

# Save experimental evidence
output_dir = project_root / "results" / "bill_amt1"
output_dir.mkdir(parents=True, exist_ok=True)

results_df.to_csv(
    output_dir / "repeated_seed_comparison.csv",
    index=False
)

print("Results saved successfully.")


Number of seeds: 20
Mean F1 improvement: 0.003559
Standard deviation: 0.00216
95% CI: (np.float64(0.002548), np.float64(0.00457))
Results saved successfully.


## Repeated-Seed Results at 15% Missingness

The experiment uses seeds 0–19 to generate 20 masks on the same validation partition. Both imputation methods use the same mask within each seed. SHML selected zero imputation in all 20 runs under the existing policy.

| Metric | Healthy | Mean Median Control | Mean Zero Imputation |
|---|---:|---:|---:|
| F1 | 0.356983 | 0.364761 | 0.368320 |
| ROC-AUC | 0.713025 | 0.711232 | 0.710679 |

Zero imputation had higher F1 than median imputation in 19 of 20 runs. The mean paired difference was +0.003559, with a standard deviation of 0.002160 and an approximate 95% confidence interval of [0.002548, 0.004570].

Zero imputation reduced ROC-AUC relative to the clean baseline in 19 of 20 runs, with a mean change of −0.002346. The F1 interval reflects variation across simulated masks on this validation partition. Generalization across independent datasets remains untested.


In [11]:

# Step 9: Repeated-seed ROC-AUC analysis
# Reuse the baseline model, scaler and validation split.

auc_results = []

for seed in range(20):
    rng = np.random.default_rng(seed)

    X_corrupted_seed = X_val.copy(deep=True)

    positions = rng.choice(
        len(X_corrupted_seed),
        size=int(len(X_corrupted_seed) * 0.15),
        replace=False
    )

    X_corrupted_seed.iloc[
        positions,
        X_corrupted_seed.columns.get_loc(FEATURE)
    ] = np.nan

    # Fixed median preprocessing control
    X_median = X_corrupted_seed.copy(deep=True)
    X_median[FEATURE] = X_median[FEATURE].fillna(
        training_median
    )

    # Zero-imputation candidate
    X_zero = X_corrupted_seed.copy(deep=True)
    X_zero[FEATURE] = X_zero[FEATURE].fillna(0)

    median_metrics = evaluate_model(
        X_batch=X_median,
        y_true=y_val,
        model=model,
        scaler=scaler
    )

    zero_metrics = evaluate_model(
        X_batch=X_zero,
        y_true=y_val,
        model=model,
        scaler=scaler
    )

    auc_results.append({
        "seed": seed,
        "healthy_auc": healthy_metrics["roc_auc"],
        "median_auc": median_metrics["roc_auc"],
        "zero_auc": zero_metrics["roc_auc"],
        "zero_vs_healthy": (
            zero_metrics["roc_auc"]
            - healthy_metrics["roc_auc"]
        ),
        "zero_vs_median": (
            zero_metrics["roc_auc"]
            - median_metrics["roc_auc"]
        )
    })

auc_df = pd.DataFrame(auc_results)

print(auc_df.round(6).to_string(index=False))

print("\nMean healthy ROC-AUC:",
      round(auc_df["healthy_auc"].mean(), 6))

print("Mean median ROC-AUC:",
      round(auc_df["median_auc"].mean(), 6))

print("Mean zero ROC-AUC:",
      round(auc_df["zero_auc"].mean(), 6))

print("Mean zero vs healthy:",
      round(auc_df["zero_vs_healthy"].mean(), 6))

print("Seeds with ROC-AUC degradation:",
      (auc_df["zero_vs_healthy"] < 0).sum(),
      "out of", len(auc_df))

# Save results
auc_df.to_csv(
    output_dir / "repeated_seed_auc_comparison.csv",
    index=False
)


 seed  healthy_auc  median_auc  zero_auc  zero_vs_healthy  zero_vs_median
    0     0.713025    0.709936  0.708495        -0.004530       -0.001441
    1     0.713025    0.711368  0.710533        -0.002493       -0.000835
    2     0.713025    0.709589  0.709182        -0.003843       -0.000407
    3     0.713025    0.710921  0.710729        -0.002296       -0.000192
    4     0.713025    0.711778  0.711597        -0.001428       -0.000181
    5     0.713025    0.712256  0.711451        -0.001575       -0.000806
    6     0.713025    0.711863  0.710479        -0.002546       -0.001383
    7     0.713025    0.712680  0.712085        -0.000941       -0.000596
    8     0.713025    0.709018  0.707769        -0.005256       -0.001249
    9     0.713025    0.710921  0.709899        -0.003126       -0.001022
   10     0.713025    0.710763  0.710624        -0.002401       -0.000139
   11     0.713025    0.709105  0.707726        -0.005299       -0.001379
   12     0.713025    0.712178  0.7121

## ROC-AUC Guardrail Sensitivity

For each of the 20 masks, calculate relative ROC-AUC loss as `(healthy ROC-AUC − candidate ROC-AUC) / healthy ROC-AUC`. Count the zero-imputation candidates that pass at maximum relative loss tolerances of 0%, 0.5%, 1%, and 2%.

This exploratory check evaluates ROC-AUC on its own. The next experiment applies it alongside the existing safety checks.


In [12]:

# Step 10: ROC-AUC guardrail sensitivity analysis
# This is a hypothetical policy analysis, not a production change.

auc_thresholds = [0.00, 0.005, 0.01, 0.02]

healthy_auc = healthy_metrics["roc_auc"]

policy_results = []

for tolerance in auc_thresholds:
    relative_auc_loss = (
        healthy_auc - auc_df["zero_auc"]
    ) / healthy_auc

    passes = relative_auc_loss <= tolerance

    policy_results.append({
        "max_relative_auc_loss": tolerance,
        "accepted": int(passes.sum()),
        "rejected": int((~passes).sum()),
        "acceptance_rate": float(passes.mean())
    })

policy_df = pd.DataFrame(policy_results)

print(policy_df.to_string(index=False))

policy_df.to_csv(
    output_dir / "auc_guardrail_sensitivity.csv",
    index=False
)


 max_relative_auc_loss  accepted  rejected  acceptance_rate
                 0.000         1        19             0.05
                 0.005        16         4             0.80
                 0.010        20         0             1.00
                 0.020        20         0             1.00


## ROC-AUC Guardrail Results

| Maximum Relative ROC-AUC Loss | Passed ROC-AUC Check | Failed ROC-AUC Check |
|---|---:|---:|
| 0% | 1 | 19 |
| 0.5% | 16 | 4 |
| 1% | 20 | 0 |
| 2% | 20 | 0 |

At a 0.5% relative loss tolerance, the ROC-AUC check flagged four zero-imputation candidates. At 1% and 2%, all 20 passed. The combined-policy experiment uses 0.5% as an exploratory setting.


In [13]:

from src.safety import evaluate_recovery_safety

auc_tolerance = 0.005
combined_results = []

for seed in range(20):
    rng = np.random.default_rng(seed)

    X_corrupted_seed = X_val.copy(deep=True)

    positions = rng.choice(
        len(X_corrupted_seed),
        size=int(len(X_corrupted_seed) * 0.15),
        replace=False
    )

    X_corrupted_seed.iloc[
        positions,
        X_corrupted_seed.columns.get_loc(FEATURE)
    ] = np.nan

    candidates = {}

    for action, fill_value in [
        ("Zero imputation", 0),
        ("Median imputation", training_median)
    ]:
        X_candidate = X_corrupted_seed.copy(deep=True)
        X_candidate[FEATURE] = (
            X_candidate[FEATURE].fillna(fill_value)
        )

        metrics = evaluate_model(
            X_batch=X_candidate,
            y_true=y_val,
            model=model,
            scaler=scaler
        )

        integrity_passed = (
            X_candidate.shape == X_corrupted_seed.shape
            and X_candidate.index.equals(X_corrupted_seed.index)
            and X_candidate.columns.equals(X_corrupted_seed.columns)
            and X_candidate.drop(columns=[FEATURE]).equals(
                X_corrupted_seed.drop(columns=[FEATURE])
            )
            and X_candidate.loc[
                X_corrupted_seed[FEATURE].notna(), FEATURE
            ].equals(
                X_corrupted_seed.loc[
                    X_corrupted_seed[FEATURE].notna(), FEATURE
                ]
            )
            and X_candidate.loc[
                X_corrupted_seed[FEATURE].isna(), FEATURE
            ].eq(fill_value).all()
            and not X_candidate.isna().any().any()
        )

        existing_safety = evaluate_recovery_safety(
            recovered_metrics=metrics,
            healthy_metrics=healthy_metrics,
            data_integrity_passed=integrity_passed
        )

        relative_auc_loss = (
            healthy_metrics["roc_auc"] - metrics["roc_auc"]
        ) / healthy_metrics["roc_auc"]

        combined_accept = (
            existing_safety["decision"] == "ACCEPT"
            and np.isfinite(relative_auc_loss)
            and relative_auc_loss <= auc_tolerance
        )

        candidates[action] = {
            "existing_accept": existing_safety["decision"] == "ACCEPT",
            "combined_accept": combined_accept,
            "auc_loss": relative_auc_loss
        }

    zero = candidates["Zero imputation"]
    median = candidates["Median imputation"]

    if zero["combined_accept"]:
        final_decision = "ZERO_ACCEPTED"
    elif median["combined_accept"]:
        final_decision = "MEDIAN_FALLBACK_ACCEPTED"
    else:
        final_decision = "ABSTAIN"

    combined_results.append({
        "seed": seed,
        "zero_existing_accept": zero["existing_accept"],
        "zero_combined_accept": zero["combined_accept"],
        "median_combined_accept": median["combined_accept"],
        "final_decision": final_decision
    })

combined_df = pd.DataFrame(combined_results)

print(combined_df.to_string(index=False))

print("\nFinal policy outcomes:")
print(combined_df["final_decision"].value_counts())

combined_df.to_csv(
    output_dir / "combined_safety_policy_results.csv",
    index=False
)


 seed  zero_existing_accept  zero_combined_accept  median_combined_accept           final_decision
    0                  True                 False                    True MEDIAN_FALLBACK_ACCEPTED
    1                  True                  True                    True            ZERO_ACCEPTED
    2                  True                 False                    True MEDIAN_FALLBACK_ACCEPTED
    3                  True                  True                    True            ZERO_ACCEPTED
    4                  True                  True                    True            ZERO_ACCEPTED
    5                  True                  True                    True            ZERO_ACCEPTED
    6                  True                  True                    True            ZERO_ACCEPTED
    7                  True                  True                    True            ZERO_ACCEPTED
    8                  True                 False                   False                  ABSTAIN
    9     

## Combined Safety Policy Evaluation

The experimental policy adds a maximum relative ROC-AUC loss of 0.5% to the existing data integrity, F1, and recall checks. It accepts zero imputation first, then tries training-median imputation as a fallback. If both candidates fail the combined checks, the policy abstains.

| Final Decision | Seeds | Share |
|---|---:|---:|
| Zero imputation accepted | 16 | 80% |
| Median fallback accepted | 2 | 10% |
| Abstention | 2 | 10% |

Zero imputation passed the existing policy in all 20 runs. The added ROC-AUC check rejected it for seeds 0, 2, 8, and 11. Median imputation passed the combined checks for seeds 0 and 2. The policy abstained for seeds 8 and 11.

These results show how the candidate selection changes when ROC-AUC is included. The 0.5% tolerance was explored offline on the same validation partition used to report these results. Independent validation of the tolerance remains open. The combined policy is implemented in this notebook experiment.
